# Wstęp do Uczenia Maszynowego - Projekt I
## Etap: Drugi Kamień Milowy 
### Autorzy: Krzysztof Osiński, Jakub Miszczak

## Import packages

In [13]:
import pandas as pd
import numpy as np
import sklearn 
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')
np.random.seed(23)
import zipfile

# Fraud Detection Transactions Dataset - Feature Extraction

In [14]:
zip_path = "fraud-detection-transactions-dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open("synthetic_fraud_dataset.csv") as file:
        df = pd.read_csv(file)

In [15]:
df.columns = df.columns.str.replace(" ","_").str.lower()
df1 = df.drop(['transaction_id','user_id'],axis='columns')
df1['timestamp'] = pd.to_datetime(df1['timestamp'])

## Variance Inflation Factor (VIF)
Wartość VIF w okolicach 1 mówi o bardzo małej współliniowości cech.

In [16]:
X = df1.drop('fraud_label', axis='columns')
Y = df1['fraud_label']

from sklearn.preprocessing import MinMaxScaler

scaled_columns = X.select_dtypes(['int64', 'float64']).columns

scaler = MinMaxScaler()

X[scaled_columns] = scaler.fit_transform(X[scaled_columns])
X.describe()

,transaction_amount,timestamp,account_balance,ip_address_flag,previous_fraudulent_activity,daily_transaction_count,avg_transaction_amount_7d,failed_transaction_count_7d,card_age,transaction_distance,risk_score,is_weekend
count,50000.0000,50000,50000.0000,50000.0000,50000.0000,50000.0000,50000.0000,50000.0000,50000.0000,50000.0000,50000.0000,50000.0000
mean,0.0847,2023-07-02 12:47:11.063999744,0.5004,0.0502,0.0984,0.4989,0.5006,0.5009,0.5000,0.4998,0.5015,0.2996
min,0.0000,2023-01-01 00:01:00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.0244,2023-04-02 03:35:30,0.2498,0.0000,0.0000,0.2308,0.2492,0.2500,0.2479,0.2513,0.2539,0.0000
50%,0.0593,2023-07-02 14:00:00,0.5014,0.0000,0.0000,0.4615,0.5022,0.5000,0.5000,0.4981,0.5022,0.0000
75%,0.1183,2023-10-01 07:13:00,0.7499,0.0000,0.0000,0.7692,0.7511,0.7500,0.7521,0.7493,0.7495,1.0000
max,1.0000,2023-12-31 23:50:00,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
std,0.0841,NaN,0.2891,0.2184,0.2979,0.3107,0.2885,0.3536,0.2899,0.2884,0.2878,0.4581


In [17]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df):
    vif_df = pd.DataFrame()
    vif_df['Column'] = df.columns
    vif_df['VIF'] = [variance_inflation_factor(df.values,i) for i in range(df.shape[1])]
    return vif_df

In [18]:
calculate_vif(X[scaled_columns])

,Column,VIF
0,transaction_amount,1.9272
1,account_balance,3.4615
2,ip_address_flag,1.0504
3,previous_fraudulent_activity,1.1036
4,daily_transaction_count,3.1596
5,avg_transaction_amount_7d,3.4722
6,failed_transaction_count_7d,2.7309
7,card_age,3.4294
8,transaction_distance,3.4529
9,risk_score,3.4787


## Weight of Evidence (WOE)

In [19]:
# Funkcja do policzenia WOE i IV dla naszych cech
def calculate_woe_iv(df, feature, target):
    
    grouped = df.groupby(feature)[target].agg(['count','sum'])
    grouped = grouped.rename(columns={'count': 'total', 'sum': 'good'})
    grouped['bad']=grouped['total']-grouped['good']
    
    total_good = grouped['good'].sum()
    total_bad = grouped['bad'].sum()
    
    grouped['good_score'] = grouped['good'] / total_good
    grouped['bad_score'] = grouped['bad'] / total_bad
    grouped['woe'] = np.log(grouped['good_score']/ grouped['bad_score'])
    grouped['iv'] = (grouped['good_score'] -grouped['bad_score'])*grouped['woe']
    
    grouped['woe'] = grouped['woe'].replace([np.inf, -np.inf], 0)
    grouped['iv'] = grouped['iv'].replace([np.inf, -np.inf], 0)
    
    total_iv = grouped['iv'].sum()
    
    return grouped, total_iv

In [20]:
iv_values = {}

# Liczymy IV 
for feature in X.columns:
    # Jeśli kolumna kategoryczna to względem kategorii
    if X[feature].dtype == 'object':
        _, iv = calculate_woe_iv(pd.concat([X, Y],axis=1), feature, 'fraud_label' )
    # Jeśli kolumna numeryczna to dzielimy na biny i liczymy względem binów
    else:
        X_binned = pd.cut(X[feature], bins=10, labels=False)
        _, iv = calculate_woe_iv(pd.concat([X_binned, Y],axis=1), feature, 'fraud_label' )
    iv_values[feature] = iv
    
# Eleganckie zapisanie wyników 
pd.set_option('display.float_format', lambda x: '{:.4f}'.format(x))

iv_df = pd.DataFrame(list(iv_values.items()), columns=['Feature', 'IV'])
iv_df = iv_df.sort_values(by='IV', ascending=False)

## Information Value (IV)
IV < 0.02  --> brak wartości predykcyjnej

0.3 <= IV < 0.5  --> silna wartość predykcyjna

IV >= 0.5  --> bardzo silna wartość predykcyjna (trzeba uważać na overfitting)

In [21]:
iv_df

,Feature,IV
11,failed_transaction_count_7d,0.5995
16,risk_score,0.4483
3,account_balance,0.0009
14,transaction_distance,0.0009
9,daily_transaction_count,0.0007
13,card_age,0.0005
2,timestamp,0.0004
10,avg_transaction_amount_7d,0.0003
15,authentication_method,0.0003
5,location,0.0002


### Spostrzeżenia
Na tym etapie wartość predykcyjną wykazują jedynie cechy 'failed_transaction_count_7d' i 'risk_score'.